# Magnetic Shielding — Interactive Simulation (sphere & transverse cylinder)

**Zero-installation version.** Run this notebook directly in the browser:

* **Google Colab:** upload this file at <https://colab.research.google.com> (or open it from the GitHub repository with the *Open in Colab* badge) and press *Runtime → Run all*. Nothing is installed on the local machine — this works even on school computers where Python installation is restricted.
* **Binder:** the repository can also be launched at <https://mybinder.org>.

The notebook solves the magnetostatic boundary-value problem for a hollow **spherical shell** and an **infinite cylindrical shell in a transverse field** (linear material, permeability $\mu = \mu_r\mu_0$, inner radius $a$, outer radius $b$, uniform applied field $H_0$), and lets you **drag sliders** for $\mu_r$ and $a/b$ to explore:

1. field-line maps with a logarithmic colour scale of $|\vec B|/(\mu_0 H_0)$ — including the small but **finite** field inside the cavity;
2. the exact shielding factor $SF = H_0/H_\mathrm{int}$ as a function of $\mu_r$, with its closed form
$$SF_\mathrm{sph}=\tfrac{2}{9}(1-k_3)\,\mu_r+\tfrac{5+4k_3}{9}+\tfrac{2(1-k_3)}{9\mu_r},\qquad
SF_\mathrm{cyl}=\tfrac{1}{4}(1-k_2)\,\mu_r+\tfrac{1+k_2}{2}+\tfrac{1-k_2}{4\mu_r},$$
with $k_3=(a/b)^3$ and $k_2=(a/b)^2$. Both reduce exactly to $SF=1$ at $\mu_r=1$.

*Companion material of the article “Magnetic shielding as a motivational tool in teaching classical electromagnetic theory”.*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import LogNorm

# ----------------------- exact analytical solutions -----------------------
def sphere_coefficients(H0, mu_r, a, b):
    k3 = (a / b) ** 3
    D = (mu_r + 2) * (2 * mu_r + 1) - 2 * k3 * (mu_r - 1) ** 2
    H_int = 9 * mu_r * H0 / D
    A = 3 * (2 * mu_r + 1) * H0 / D
    B = 3 * (mu_r - 1) * a ** 3 * H0 / D
    C = b ** 3 * (mu_r - 1) * (2 * mu_r + 1) * (1 - k3) * H0 / D   # induced dipole
    return H_int, H0 / H_int, A, B, C

def cylinder_coefficients(H0, mu_r, a, b):
    D = (mu_r + 1) ** 2 * b ** 2 - (mu_r - 1) ** 2 * a ** 2
    H_int = 4 * mu_r * b ** 2 * H0 / D
    A = 2 * (mu_r + 1) * b ** 2 * H0 / D
    B = 2 * (mu_r - 1) * a ** 2 * b ** 2 * H0 / D
    C = b ** 2 * (mu_r ** 2 - 1) * (b ** 2 - a ** 2) * H0 / D
    return H_int, H0 / H_int, A, B, C

def field_on_grid(X, Y, geometry, H0, mu_r, a, b):
    R, T = np.hypot(X, Y), np.arctan2(Y, X)
    Hx, Hy = np.zeros_like(X), np.zeros_like(Y)
    mu_map = np.ones_like(X)
    cav, sh, ext = R < a, (R >= a) & (R <= b), R > b
    mu_map[sh] = mu_r
    coeff = sphere_coefficients if geometry == "sphere" else cylinder_coefficients
    H_int, SF, A, B, C = coeff(H0, mu_r, a, b)
    p = 3 if geometry == "sphere" else 2          # multipole power
    q = 2 if geometry == "sphere" else 1          # exterior dipole power factor
    Hx[cav], Hy[cav] = H_int, 0.0
    r, t = R[sh], T[sh]
    Hr = (A - (p - 1) * B / r ** p) * np.cos(t)
    Ht = -(A + B / r ** p) * np.sin(t)
    Hx[sh] = Hr * np.cos(t) - Ht * np.sin(t)
    Hy[sh] = Hr * np.sin(t) + Ht * np.cos(t)
    r, t = R[ext], T[ext]
    Hr = (H0 + q * C / r ** p) * np.cos(t)
    Ht = -(H0 - C / r ** p) * np.sin(t)
    Hx[ext] = Hr * np.cos(t) - Ht * np.sin(t)
    Hy[ext] = Hr * np.sin(t) + Ht * np.cos(t)
    return Hx, Hy, mu_map * np.hypot(Hx, Hy), H_int, SF

def plot_fields(mu_r=1000.0, ratio=0.5, H0=1.0, b=2.0, L=4.0, N=350):
    """Two-panel field map. ratio = a/b."""
    a = ratio * b
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.8))
    x = np.linspace(-L, L, N); X, Y = np.meshgrid(x, x)
    for ax, geom, ti in zip(axes, ("sphere", "cylinder"),
                            ("Spherical shell (meridian plane)",
                             "Cylindrical shell (transverse plane)")):
        Hx, Hy, Bmag, H_int, SF = field_on_grid(X, Y, geom, H0, mu_r, a, b)
        vmin = max(H_int / 3, 1e-7)
        vmax = float(np.percentile(Bmag / H0, 99.5))
        pcm = ax.pcolormesh(X, Y, Bmag / H0, norm=LogNorm(vmin=vmin, vmax=vmax),
                            cmap="viridis", shading="auto")
        # cavity masked out of the density-based pass: equal line density
        # everywhere would visually overstate the weak interior field
        mask_cav = np.sqrt(X**2 + Y**2) < a
        Hx_o = np.ma.array(Hx, mask=mask_cav); Hy_o = np.ma.array(Hy, mask=mask_cav)
        ax.streamplot(X, Y, Hx_o, Hy_o, density=1.1, color="w", linewidth=0.9)
        # three thin seeded lines inside the cavity: small but finite field
        Hx_i = np.ma.array(Hx, mask=~mask_cav); Hy_i = np.ma.array(Hy, mask=~mask_cav)
        seeds = np.column_stack([np.zeros(3), np.linspace(-0.55 * a, 0.55 * a, 3)])
        try:
            ax.streamplot(X, Y, Hx_i, Hy_i, start_points=seeds, color="w",
                          linewidth=0.7, broken_streamlines=False)
        except TypeError:
            ax.streamplot(X, Y, Hx_i, Hy_i, start_points=seeds, color="w", linewidth=0.7)
        for rr in (a, b):
            ax.add_patch(Circle((0, 0), rr, fill=False, ec="k", lw=1.3, zorder=4))
        ax.set(xlim=(-L, L), ylim=(-L, L), xlabel="x (m)", ylabel="y (m)",
               title=f"{ti}  —  $\\mu_r={mu_r:g}$")
        ax.set_aspect("equal")
        ax.text(0.03, 0.965, f"$H_{{int}}/H_0={H_int/H0:.2e}$\n$SF={SF:.1f}$",
                transform=ax.transAxes, va="top", fontsize=10,
                bbox=dict(fc="white", alpha=0.85, boxstyle="round,pad=0.3"))
        fig.colorbar(pcm, ax=ax, shrink=0.85,
                     label=r"$|\vec B|/(\mu_0 H_0)$ (log)")
    plt.tight_layout(); plt.show()


In [ ]:
# Static example — reproduces Fig. 1 of the article
plot_fields(mu_r=1000.0, ratio=0.5)

In [ ]:
# Interactive exploration: drag the sliders (works in Colab and Jupyter).
try:
    from ipywidgets import interact, FloatLogSlider, FloatSlider
    interact(plot_fields,
             mu_r=FloatLogSlider(value=1000, base=10, min=0, max=5, step=0.1,
                                 description=r"mu_r"),
             ratio=FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05,
                               description="a/b"),
             H0=(0.5, 2.0, 0.5), b=(1.0, 3.0, 0.5),
             L=(3.0, 6.0, 1.0), N=(150, 500, 50));
except ImportError:
    print("ipywidgets not available - showing the static figure instead.")
    plot_fields()

In [ ]:
# Exact shielding factor SF = H0/H_int versus mu_r (log-log)
def SF_sphere(mu_r, a, b):
    k3 = (a / b) ** 3
    return ((mu_r + 2) * (2 * mu_r + 1) - 2 * k3 * (mu_r - 1) ** 2) / (9 * mu_r)

def SF_cylinder(mu_r, a, b):
    k2 = (a / b) ** 2
    return ((mu_r + 1) ** 2 - k2 * (mu_r - 1) ** 2) / (4 * mu_r)

def plot_SF(ratio=0.5, b=2.0):
    a = ratio * b
    k3, k2 = ratio ** 3, ratio ** 2
    mu = np.logspace(0, np.log10(5e4), 500)
    fig, ax = plt.subplots(figsize=(8.2, 5.6))
    ax.loglog(mu, SF_sphere(mu, a, b), "b-", lw=2.6, label="sphere (exact)")
    ax.loglog(mu, SF_cylinder(mu, a, b), "g-", lw=2.6,
              label="cylinder, transverse (exact)")
    m = mu > 30
    ax.loglog(mu[m], 2 / 9 * (1 - k3) * mu[m], "b--", lw=1.1,
              label=r"$\frac{2}{9}(1-k_3)\mu_r$")
    ax.loglog(mu[m], 0.25 * (1 - k2) * mu[m], "g--", lw=1.1,
              label=r"$\frac{1}{4}(1-k_2)\mu_r$")
    ax.axhline(1, color="gray", ls=":", lw=1)
    ax.set(xlabel=r"$\mu_r$", ylabel=r"$SF = H_0/H_{int}$",
           title=f"Shielding factor, a/b = {ratio:.2f}")
    ax.grid(True, which="both", ls="--", alpha=0.3)
    ax.legend(loc="upper left")
    plt.tight_layout(); plt.show()

try:
    from ipywidgets import interact, FloatSlider
    interact(plot_SF, ratio=FloatSlider(value=0.5, min=0.05, max=0.95,
                                        step=0.05, description="a/b"),
             b=(1.0, 3.0, 0.5));
except ImportError:
    plot_SF()

## Physics summary

With no free currents, $\nabla\times\vec H=0$ allows $\vec H=-\nabla\phi_m$, and $\nabla\cdot\vec B=0$ with $\vec B=\mu\vec H$ gives $\nabla^2\phi_m=0$ in each region. Matching the continuity of $B_n$ and $H_t$ at $r=a$ and $r=b$, only the $l=1$ (sphere) / $\nu=1$ (cylinder) terms survive, and the cavity field is **uniform**:

$$\vec H_\mathrm{int}^{\,\mathrm{sph}}=\frac{9\mu_r H_0}{(\mu_r+2)(2\mu_r+1)-2\frac{a^3}{b^3}(\mu_r-1)^2}\,\hat z,
\qquad
\vec H_\mathrm{int}^{\,\mathrm{cyl}}=\frac{4\mu_r b^2 H_0}{(\mu_r+1)^2b^2-(\mu_r-1)^2a^2}\,\hat x .$$

For $\mu_r\gg1$: $H_\mathrm{int}^\mathrm{sph}\approx\dfrac{9H_0}{2\mu_r\!\left(1-\frac{a^3}{b^3}\right)}$ and $H_\mathrm{int}^\mathrm{cyl}\approx\dfrac{4H_0\,b^2}{\mu_r\,(b^2-a^2)}$ — the $1/\mu_r$ signature of passive shielding.

**References.** J. D. Jackson, *Classical Electrodynamics*, 3rd ed. (Wiley, 1999), Sec. 5.12; D. J. Griffiths, *Introduction to Electrodynamics*, 5th ed. (Cambridge, 2023); A. J. Mager, IEEE Trans. Magn. **6**, 67 (1970); J. F. Hoburg, IEEE Trans. Electromagn. Compat. **37**, 574 (1995); T. J. Sumner, J. M. Pendlebury and K. F. Smith, J. Phys. D **20**, 1095 (1987).

---
*Code developed by the authors with the assistance of an AI tool (Anthropic Claude) for drafting and refactoring; all equations and outputs were derived, verified and validated by the authors, who take full responsibility for the content. Released under the MIT License.*
